In [ ]:
!pip install ultralytics

In [1]:
import os
import json
import zipfile
from pathlib import Path

import pandas as pd
from ultralytics import YOLO

In [3]:
dataset_path = './ai09-level1-project.zip'
output_dir = './ai09-project01/'
os.makedirs(output_dir, exist_ok=True)

with zipfile.ZipFile(dataset_path) as zip_file:
    zip_file.extractall(path=output_dir)

data_root = "./ai09-project01/sprint_ai_project1_data"
train_image_dir = os.path.join(data_root, "train_images")
train_annotation_dir = os.path.join(data_root, "train_annotations")
test_image_dir = os.path.join(data_root, "test_images")

print("Train image dir:", train_image_dir)
print("Train annotation dir:", train_annotation_dir)
print("Test image dir:", test_image_dir)

Train image dir: ./ai09-project01/sprint_ai_project1_data\train_images
Train annotation dir: ./ai09-project01/sprint_ai_project1_data\train_annotations
Test image dir: ./ai09-project01/sprint_ai_project1_data\test_images


In [2]:
BEST_PT_PATH = "./best_yolo26m.pt"

DATA_ROOT = "./ai09-project01/sprint_ai_project1_data/"
TRAIN_ANNOTATION_DIR = os.path.join(DATA_ROOT, "train_annotations")
TEST_IMAGE_DIR = os.path.join(DATA_ROOT, "test_images")

SUBMISSION_CSV_PATH = "./submission.csv"

CONF_TH = 0.5
IOU_TH = 0.5
MAX_DET = 4
DEVICE = 0  # GPU: 0, CPU: "cpu"

In [3]:
assert os.path.exists(BEST_PT_PATH), f"best.pt not found: {BEST_PT_PATH}"
assert os.path.isdir(TRAIN_ANNOTATION_DIR), f"train annotation dir not found: {TRAIN_ANNOTATION_DIR}"
assert os.path.isdir(TEST_IMAGE_DIR), f"test image dir not found: {TEST_IMAGE_DIR}"

image_paths = sorted([
    p for p in Path(TEST_IMAGE_DIR).glob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
])

print("best.pt:", BEST_PT_PATH)
print("train annotation dir:", TRAIN_ANNOTATION_DIR)
print("test image dir:", TEST_IMAGE_DIR)
print("num test images:", len(image_paths))
print("sample test images:", [p.name for p in image_paths[:5]])

best.pt: ./best_yolo26m.pt
train annotation dir: ./ai09-project01/sprint_ai_project1_data/train_annotations
test image dir: ./ai09-project01/sprint_ai_project1_data/test_images
num test images: 842
sample test images: ['1.png', '10.png', '100.png', '1003.png', '1004.png']


In [4]:
def safe_get(d, keys, default=None):
    for k in keys:
        if k in d and d[k] not in [None, ""]:
            return d[k]
    return default

def parse_annotation_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []
    if isinstance(data, dict) and "annotations" in data:
        categories = data.get("categories", [])
        category_map = {}
        for cat in categories:
            cat_id = safe_get(cat, ["id"], default=None)
            cat_name = safe_get(cat, ["name"], default=None)
            category_map[cat_id] = cat_name

        for ann in data.get("annotations", []):
            category_id = safe_get(ann, ["category_id"], default=None)
            rows.append({
                "category_id": category_id,
                "category_name": category_map.get(category_id, None),
            })
        return rows

    raise ValueError(f"Unsupported annotation format: {json_path}")

def build_name_to_original_category_id(annotation_root):
    rows = []
    for root, dirs, files in os.walk(annotation_root):
        for jf in files:
            if not jf.lower().endswith(".json"):
                continue
            json_path = os.path.join(root, jf)
            try:
                rows.extend(parse_annotation_json(json_path))
            except Exception as e:
                print(f"[WARNING] failed to parse {json_path}: {e}")

    ann_df = pd.DataFrame(rows)
    ann_df = ann_df.dropna(subset=["category_name", "category_id"]).drop_duplicates()
    name_to_original_category_id = dict(zip(ann_df["category_name"].astype(str), ann_df["category_id"].astype(int)))

    print("num original category names:", len(name_to_original_category_id))
    sample_items = list(name_to_original_category_id.items())[:5]
    print("sample name -> original category_id:", sample_items)
    return name_to_original_category_id

name_to_original_category_id = build_name_to_original_category_id(TRAIN_ANNOTATION_DIR)

model = YOLO(BEST_PT_PATH)
print("model loaded.")
print("num model classes:", len(model.names))
print("sample model.names:", list(model.names.items())[:5])

model_idx_to_original_category_id = {}
missing_names = []

for idx, class_name in model.names.items():
    class_name = str(class_name)
    if class_name in name_to_original_category_id:
        model_idx_to_original_category_id[int(idx)] = int(name_to_original_category_id[class_name])
    else:
        missing_names.append((int(idx), class_name))

print("mapped model classes:", len(model_idx_to_original_category_id))
print("sample model_idx -> original category_id:", list(model_idx_to_original_category_id.items())[:10])

if missing_names:
    print("[WARNING] missing class names in annotation mapping:")
    print(missing_names[:20])

num original category names: 56
sample name -> original category_id: [('보령부스파정 5mg', 1900), ('가바토파정 100mg', 16548), ('스토가정 10mg', 19607), ('레일라정', 29451), ('신바로정', 33009)]
model loaded.
num model classes: 56
sample model.names: [(0, '가바토파정 100mg'), (1, '글리아타민연질캡슐'), (2, '글리틴정(콜린알포세레이트)'), (3, '기넥신에프정(은행엽엑스)(수출용)'), (4, '노바스크정 5mg')]
mapped model classes: 56
sample model_idx -> original category_id: [(0, 16548), (1, 32310), (2, 33880), (3, 3483), (4, 19861), (5, 24850), (6, 3832), (7, 12778), (8, 16551), (9, 21771)]


In [5]:
rows = []
annotation_id = 1
unmapped_pred_classes = set()

for idx, img_path in enumerate(image_paths, start=1):
    result = model.predict(
        source=str(img_path),
        conf=CONF_TH,
        iou=IOU_TH,
        max_det=MAX_DET,
        device=DEVICE,
        verbose=False,
        save=False
    )[0]

    image_id = int(img_path.stem)

    if result.boxes is None or len(result.boxes) == 0:
        continue

    xyxy = result.boxes.xyxy.cpu().numpy()
    cls = result.boxes.cls.cpu().numpy().astype(int)
    conf = result.boxes.conf.cpu().numpy()

    for box, pred_cls, score in zip(xyxy, cls, conf):
        pred_cls = int(pred_cls)

        if pred_cls not in model_idx_to_original_category_id:
            unmapped_pred_classes.add(pred_cls)
            continue

        x1, y1, x2, y2 = box
        original_category_id = int(model_idx_to_original_category_id[pred_cls])

        rows.append({
            "annotation_id": annotation_id,
            "image_id": image_id,
            "category_id": original_category_id,
            "bbox_x": max(0, int(round(x1))),
            "bbox_y": max(0, int(round(y1))),
            "bbox_w": max(0, int(round(x2 - x1))),
            "bbox_h": max(0, int(round(y2 - y1))),
            "score": float(score),
        })
        annotation_id += 1

    if idx % 100 == 0 or idx == len(image_paths):
        print(f"[{idx}/{len(image_paths)}] processed")

if unmapped_pred_classes:
    print("[WARNING] unmapped predicted classes:", sorted(unmapped_pred_classes))

[100/842] processed
[200/842] processed
[300/842] processed
[400/842] processed
[500/842] processed
[600/842] processed
[700/842] processed
[800/842] processed
[842/842] processed


In [6]:
submission_df = pd.DataFrame(rows, columns=[
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "score",
])

submission_df = submission_df.sort_values(
    by=["image_id", "score"],
    ascending=[True, False]
).reset_index(drop=True)

submission_df["annotation_id"] = range(1, len(submission_df) + 1)

submission_df.to_csv(SUBMISSION_CSV_PATH, index=False)

print("saved:", SUBMISSION_CSV_PATH)
print("num rows:", len(submission_df))
submission_df.head()

saved: ./submission.csv
num rows: 2555


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,27926,596,677,259,478,0.972563
1,2,1,1900,158,250,203,127,0.929472
2,3,1,16551,596,677,259,478,0.566717
3,4,3,27926,568,631,264,484,0.976711
4,5,3,1900,140,241,199,129,0.893038


In [7]:
print("category_id range:", submission_df["category_id"].min(), submission_df["category_id"].max())
print("bbox_x min:", submission_df["bbox_x"].min(), "bbox_y min:", submission_df["bbox_y"].min())
print("num unique category_ids:", submission_df["category_id"].nunique())
print("sample rows:")
display(submission_df.head())

category_id range: 1900 41768
bbox_x min: 0 bbox_y min: 0
num unique category_ids: 56
sample rows:


,annotation_id,image_id,category_id,bbox_x,bbox_y,bbox_w,bbox_h,score
0,1,1,27926,596,677,259,478,0.972563
1,2,1,1900,158,250,203,127,0.929472
2,3,1,16551,596,677,259,478,0.566717
3,4,3,27926,568,631,264,484,0.976711
4,5,3,1900,140,241,199,129,0.893038


In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

In [ ]:
VIS_SAVE_DIR = "/content/test_pred_vis"

os.makedirs(VIS_SAVE_DIR, exist_ok=True)

In [ ]:
def visualize_test_predictions(
    model,
    test_image_dir,
    save_dir,
    num_samples=10,
    conf=0.5,
    iou=0.75,
    show_inline=True,
    random_sample=True
):
    image_paths = sorted([
        str(p) for p in Path(test_image_dir).glob("*")
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ])

    if len(image_paths) == 0:
        print("No test images found.")
        return

    if random_sample:
        selected_paths = random.sample(image_paths, min(num_samples, len(image_paths)))
    else:
        selected_paths = image_paths[:num_samples]

    print(f"Selected {len(selected_paths)} images for visualization.")

    for img_path in selected_paths:
        results = model.predict(
            source=img_path,
            conf=conf,
            iou=iou,
            save=False,
            verbose=False
        )

        result = results[0]

        # ultralytics 기본 plot 결과 (BGR)
        plotted = result.plot()

        # 저장
        save_path = os.path.join(save_dir, Path(img_path).name)
        cv2.imwrite(save_path, plotted)

        if show_inline:
            plt.figure(figsize=(8, 8))
            plt.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
            plt.title(Path(img_path).name)
            plt.axis("off")
            plt.show()

    print(f"Saved visualizations to: {save_dir}")

In [ ]:
visualize_test_predictions(
    model=model,
    test_image_dir=TEST_IMAGE_DIR,
    save_dir=VIS_SAVE_DIR,
    num_samples=10,
    conf=0.5,
    iou=0.75,
    show_inline=True,
    random_sample=True
)

In [ ]:
visualize_test_predictions(
    model=model,
    test_image_dir=TEST_IMAGE_DIR,
    save_dir=VIS_SAVE_DIR,
    num_samples=10,
    conf=0.5,
    iou=0.75,
    show_inline=True,
    random_sample=True
)